In [ ]:
import os
import xarray as xr
import numpy as np
from cdo import Cdo
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import tempfile
import logging

from eocene.nemo.eoceneNEMO import EoceneNEMO

cdo = Cdo()
loggy = logging.getLogger(__name__)

In [ ]:
eocene = EoceneNEMO(
    herold_folder="/lus/h2resw01/hpcperm/ecme3497/herold-et-al-2014/Supp",
    input_folder="/lus/h2resw01/hpcperm/ecme3497/ece4-data-v9/ece-4-inidata_4.1.9",
    output_folder="/lus/h2resw01/hpcperm/ecme3497/ece4-data-v9-paleo/ece-4-inidata_4.1.9"
)

grid= "r720x360"

In [ ]:
m2_varname="eo_tidal_dissipation"
hcri_value=500.0
cleanup=True

herold_file_orig = os.path.join(eocene.herold_folder, "Green_Huber_eocene_tidal_dissipation_1x1.nc")
herold_file_remap = os.path.join(eocene.herold_folder, "Green_Huber_eocene_tidal_dissipation_r720x360.nc")
herold_bath_orig = os.path.join(eocene.herold_folder, "herold_etal_eocene_topo_1x1.nc")
herold_bath_remap = os.path.join(eocene.herold_folder, "herold_etal_eocene_topo_1x1_r720x360.nc")
zdfiwm_file = os.path.join(eocene.input_folder, "nemo/initial/zdfiwm_forcing_r720x360.nc")
output_file = os.path.join(eocene.output_folder, "nemo/initial/zdfiwm_forcing_r720x360.nc")
runoff = os.path.join(eocene.input_folder, "nemo/climatology/runoff_DaiTrenberth_PALEORCA2.nc")
runoff_output = os.path.join(eocene.output_folder, "nemo/climatology/runoff_DaiTrenberth_PALEORCA2.nc")

In [ ]:
if cleanup and os.path.exists(output_file):
    os.remove(output_file)
    loggy.info("Cleaning file: %s", output_file)

herold_file_remap = eocene.cdo.remapcon(eocene.grid, input=herold_file_orig)
herold_bath_remap = eocene.cdo.remapcon(eocene.grid, input=herold_bath_orig)

ds_m2 = xr.open_dataset(herold_file_remap)
ds_bt = xr.open_dataset (herold_bath_remap)
ds_pd = xr.open_dataset(zdfiwm_file)

m2 = ds_m2[m2_varname]

In [ ]:
# Interpolate topo onto the ds_pd grid
topo_interp = ds_bt["topo"].interp(
    lat=ds_pd.latitude,
    lon=ds_pd.longitude,
    method="linear"
)

In [ ]:
ocean_topo = topo_interp.where(topo_interp < 0.0)

In [ ]:
import matplotlib.pyplot as plt

ocean_topo.plot(figsize=(12, 6))
plt.title("Negative topo values (bathymetry)")
plt.show()

In [ ]:
# make a copy of present-day file
ds_out = ds_pd.copy(deep=True)

# inject M2 into power_cri (1/3 only)
ds_out['power_cri'][:] = (1.0 / 3.0) * m2.values

# modify bathymetry
ds_out['bathymetry'][:] = ocean_topo

# zero in other energy reservoirs
ds_out['power_nsq'][:] = 0.0
ds_out['power_sho'][:] = 0.0
ds_out['power_bot'][:] = 0.0

# set vertical length scales
ds_out['scale_cri'][:] = hcri_value
ds_out['scale_bot'][:] = 0.0   # unused if power_bot = 0

# convert NaN into zeroes
ds_out = ds_out.fillna(0)

# write output file
os.makedirs(os.path.join(eocene.output_folder, "nemo/initial"), exist_ok=True)
ds_out.to_netcdf(output_file)
loggy.info("Written NEMO 4.2 tidal file: %s", output_file)

ds_m2.close()
ds_pd.close()

In [ ]:
print (ds_out)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
ds_out["bathymetry"].plot(cmap="Blues_r", vmin=-6000, vmax=0)
plt.title("Output bathymetry")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
ds_pd["bathymetry"].plot(cmap="Blues_r", vmin=6000, vmax=0)
plt.title("Output bathymetry")
plt.show()

In [ ]:
ds_rnf = xr.open_dataset(runoff)

In [ ]:
print (ds_rnf)

In [ ]:
ds_out = ds_rnf.copy(deep=True)
ds_out['socoefr'][:] = 0.0
ds_out['sorunoff'][:] = 0.0

In [ ]:
# convert NaN into zeroes
ds_out = ds_out.fillna(0)

# write output file
os.makedirs(os.path.join(eocene.output_folder, "nemo/initial"), exist_ok=True)
ds_out.to_netcdf(runoff_output)
loggy.info("Written NEMO 4.2 runoff file: %s", output_file)

ds_rnf.close()